## **1.Import Libraries**

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:

def change_directory(folder_name):
    
    current_path = os.getcwd()

    parent_path = os.path.dirname(current_path)

    # Change directory
    os.chdir(parent_path)
    new_path = os.path.join(parent_path,folder_name)
    if os.path.exists(new_path):
        os.chdir(new_path)
        print("Now in:", os.getcwd())
    else:
        print("Directory not found:", new_path)

In [ ]:
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
from transformers import DataCollatorWithPadding
from transformers import AutoTokenizer,TrainingArguments, Trainer,AutoConfig,AutoModelForSequenceClassification,pipeline
import evaluate
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import wandb
from transformers import TrainingArguments, Trainer,AutoConfig
from sentence_transformers import SentenceTransformer
import numpy as np
from huggingface_hub import notebook_login
from datasets import Dataset
from  transformers_preprocessing import *
from dotenv import load_dotenv



## **2. Load dataset**

In [ ]:
change_directory("data")

In [ ]:
df_train = pd.read_csv('train.csv')

df_test = pd.read_csv('test.csv')

In [ ]:
df_train['text'] = clean(df_train['text'])

df_train

In [ ]:
df_test['text'] = clean(df_test['text'])

df_test.drop(columns=['id'], inplace=True)

df_test

In [ ]:
y =  df_train['label']

y

In [ ]:
list_set = df_train['text'].tolist()

In [ ]:
list_set = [str(i) for i in list_set]

In [ ]:
sentences_df = pd.DataFrame({'sentence': list_set})

# Combine into one DataFrame
data = pd.concat([sentences_df, y], axis=1)


## **3. Train test split**

In [ ]:
train_df, test_df, label_train,label_test = train_test_split(data["sentence"],data["label"],test_size=0.2, random_state=42, stratify=data['label'])

In [ ]:
train_df = train_df.reset_index(drop=True)

test_df = test_df.reset_index(drop=True)

label_train = label_train.reset_index(drop=True)

label_test= label_test.reset_index(drop=True)



## **4. Models**

### **4.1 Model with Encoder Embeddings + Traditional Classifiers**

In [ ]:

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Convert text to embeddings
train_embeddings = model.encode(train_df, show_progress_bar=True)

test_embeddings = model.encode(test_df, show_progress_bar=True)

#### 4.1.1 Logistic Regression

In [ ]:

# Train a Logistic Regression on our train embeddings
clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, label_train)

In [ ]:

# Predict previously unseen instances
y_pred = clf.predict(test_embeddings)
labels = {"Negative":0, "Positive":1,"Neutral":2}
print(classification_report(y_pred, label_test, target_names = labels.keys()))

#### 4.1.2 Support Vector Machines

In [ ]:
svm_model = SVC(random_state=42)
svm_model.fit(train_embeddings, label_train)

In [ ]:
# Predict previously unseen instances
y_pred = svm_model.predict(test_embeddings)
labels = {"Negative":0, "Positive":1,"Neutral":2}
print(classification_report(y_pred, label_test, target_names = labels.keys()))

#### 4.1.3 Multi Layer Perceptron

In [ ]:
mlp_model = MLPClassifier(random_state=42)
mlp_model.fit(train_embeddings, label_train)

In [ ]:
# Predict previously unseen instances
y_pred = mlp_model.predict(test_embeddings)
labels = {"Negative":0, "Positive":1,"Neutral":2}
print(classification_report(y_pred, label_test, target_names = labels.keys()))

### **4.2 BERT Models (with encoding and classifier embedded )**

In [ ]:
train_df = pd.concat([train_df.reset_index(drop=True), label_train], axis=1)

test_df = pd.concat([test_df.reset_index(drop=True), label_test], axis=1)

In [ ]:
labels = {"Negative":0, "Positive":1,"Neutral":2}

### **4.2.1 Bertweet-base-sentiment-analysis**

In [ ]:

# Set up the inference pipeline using a model from the 🤗 Hub
sentiment_analysis = pipeline(model="finiteautomata/bertweet-base-sentiment-analysis")


sentiment_analysis("i love this computer")



In [ ]:
predict=[]
for sentence in tqdm(test_df['sentence'].values.tolist()):
    sentiment = sentiment_analysis(str(sentence))
    top_label = sentiment[0]['label']

    # Map label to class index
    if top_label == 'NEG':
        predict.append(0)  # negative
    elif top_label == 'NEU':
        predict.append(2)  # neutral
    elif top_label == 'POS':
        predict.append(1)  # positive


In [ ]:

print(classification_report(predict, test_df['label'].values.tolist(), target_names = labels.keys()))

In [ ]:
train_df

### **4.2.2 Fine tuning a pre trained model**

In [ ]:


def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    labels = p.label_ids
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
        "precision": precision_score(labels, preds, average="weighted"),
        "recall": recall_score(labels, preds, average="weighted"),
    }

Insert the below token on notebook login

In [ ]:
token= os.getenv("HF_TOKEN")

token


In [ ]:
notebook_login()

In [ ]:
model_path = 'ProsusAI/finbert'

In [ ]:
label2id = {label: idx for idx, label in labels.items()}

config = AutoConfig.from_pretrained(
    model_path,
    num_labels=len(labels),
    id2label=label2id,
    label2id=labels
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [ ]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
train_dataset

In [ ]:
def preprocess_function(examples):
   return tokenizer(examples['sentence'], truncation=True)

tokenized_train = train_dataset.map(preprocess_function)
tokenized_test = test_dataset.map(preprocess_function)

In [ ]:
tokenized_train

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
labels = {"negative":0, "Positive":1,"Neutral":2}

#### **4.2.2.1 Create Model on Hugging Face**

In [ ]:

model = AutoModelForSequenceClassification.from_pretrained(model_path,config=config)


repo_name = "finetuning-sentiment-model"

training_args = TrainingArguments(
   output_dir=repo_name,
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=2,
   weight_decay=0.01,
   save_strategy="epoch",
   push_to_hub=True,
)

trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)



Insert trainer_key on train method

In [ ]:
trainer_key=os.getenv("TRAIN_TOKEN")

trainer_key

In [ ]:

# Initialize a new W&B run
wandb.init(project="finetuning-sentiment-model")

train_result = trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
trainer.push_to_hub()

#### **4.2.2.2 Use our fined tuned model**



Fine tunning of Finbert available at [Hugging Face Model](https://huggingface.co/ruitj/finetuning-sentiment-model)

In [ ]:
sentiment_model = pipeline(model="ruitj/finetuning-sentiment-model")

sentiment_model(["futures are up"])



In [ ]:
predict=[]
for sentence in tqdm(test_df['sentence'].values.tolist()):
    sentiment = sentiment_model(str(sentence))
    top_label = sentiment[0]['label']

    # Map label to class index
    if top_label == 'Positive':
        predict.append(1)  # negative
    elif top_label == 'Negative':
        predict.append(0)  # neutral
    elif top_label == 'Neutral':
        predict.append(2)  # positive


In [ ]:

print(classification_report(predict,test_df['label'].values.tolist() , target_names = labels.keys()))

In [ ]:
predict=[]
for sentence in tqdm(df_test['text'].values.tolist()):
    sentiment = sentiment_model(str(sentence))
    top_label = sentiment[0]['label']

    # Map label to class index
    if top_label == 'Positive':
        predict.append(1)  # negative
    elif top_label == 'Negative':
        predict.append(0)  # neutral
    elif top_label == 'Neutral':
        predict.append(2)  # positive


In [ ]:
test_final = pd.concat([df_test['text'].reset_index(drop=True), pd.DataFrame(predict)], axis=1)

In [ ]:
change_directory("predictions")

In [ ]:
test_final.to_csv('predictions_tests_01_31.csv')